# Protocole FAMOUS

### Run the cells one by one up to the "Other experiments" heading

Inference for all conditions. Run from the root of the supplementary material; models are served with Ollama (the first cell installs it and pulls the five models). Runs were made on Kaggle with two T4 GPUs. The English 800-character condition uses `construire_prompt` with `texte[:800]`; the 2,000-character condition changes the slice to `texte[:2000]`; the French condition uses `construire_prompt_fr`. `extraire_verdict` is the keyword rule that maps responses to verdicts.


In [ ]:
!pip install -q ddgs ollama
!apt-get install -y zstd -qq
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q statsmodels


!pkill -f "ollama serve"
import time
time.sleep(2)

import subprocess, time

with open('ollama.log', 'w') as log_file:
    subprocess.Popen(['ollama', 'serve'], stdout=log_file, stderr=log_file)

time.sleep(5)
print("Service Ollama demarre")

# Gemma 3
!ollama pull gemma3:4b

# Qwen 2.5
!ollama pull qwen2.5:7b

# Llama 3.2
!ollama pull llama3.2:1b
!ollama pull llama3.2:3b

# Phi-4
!ollama pull phi4-mini


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import csv
import random
import re
import time
import ollama
from ddgs import DDGS
from collections import Counter
from statsmodels.stats.contingency_tables import cochrans_q
from scipy.stats import chi2
import numpy as np
import os, datetime
from urllib.parse import urlparse

In [ ]:
def lire_csv(chemin):
    with open(chemin, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

In [ ]:
# paths relative to the root of the supplementary material
chemin_in = "data/"
chemin_pools = "data/domain_pools.csv"
chemin_tirage = "data/tirages_7.csv"   # same content as data/source_sets.csv

chemin_out = "out/"
os.makedirs(chemin_out, exist_ok=True)

articles = lire_csv(chemin_in + "articles_150.csv")   # fixed sample used in all experiments

# curated pools by cell (same files, in the same sorted order, as categorie_domaine/domaines_X.csv)
pools = lire_csv(chemin_pools)
def cellule(notoriete, credibilite):
    return [d for d in pools if d["notoriete"] == notoriete and d["credibilite"] == credibilite]
catA = cellule("s0_connue", "credible")
catB = cellule("s0_connue", "peu_credible")
catC = cellule("s1_peu_connue", "credible")
catD = cellule("s1_peu_connue", "peu_credible")

print(f"Sample  : {len(articles)} articles")
print(f"Cat A  : {len(catA)} domains")
print(f"Cat B  : {len(catB)} domains")
print(f"Cat C  : {len(catC)} domains")
print(f"Cat D  : {len(catD)} domains")


In [ ]:
# cell to run only if we want to use the RAG evidence from the file
CHEMIN_PREUVES = "data/evidence_150.csv"   # evidence retrieved on 9 October 2026

lignes_preuves = lire_csv(CHEMIN_PREUVES)

preuves_csv = {r["id"]: r["Preuves"] for r in lignes_preuves}

for a in articles:
    a["preuves"] = preuves_csv[a["id"]]

print(len(lignes_preuves), "lignes lues,", len(preuves_csv), "preuves rechargées")

In [ ]:
def chercher_preuves_2(requete, max_tentatives=3):
    # searches the web for information about the text passed as a parameter
    # returns the first 3 results found, formatted as a bullet list
    # if finding evidence fails, we retry up to 3 times


    for tentative in range(1, max_tentatives + 1):
        try:
            # timeout increased to 20s (instead of the default 5s)
            with DDGS(timeout=20) as ddgs:
                resultats = ddgs.text(query=requete, max_results=3)

            if not resultats:
                return "(no evidence found)", []

            morceaux = []
            domaines = []
            
            for r in resultats:
                morceaux.append(f"- {r.get('title', '')} : {r.get('body', '')[:200]}")
                domaines.append(urlparse(r.get("href", "")).netloc)
            return "\n".join(morceaux), domaines

        except Exception as e:
            print(f"  [!] Attempt {tentative}/{max_tentatives} failed : {e}")
            if tentative < max_tentatives:
                # wait longer and longer after each attempt (3s, 6s, 9s)
                time.sleep(3 * tentative)

    # if all attempts failed
    return "(no evidence found after several attempts)", []


In [ ]:
def construire_prompt_fr(titre, texte, source, preuves, avec_rag):
    texte_court = texte[:800]

    prompt = "Voici un article a verifier.\n\n"
    if source != "":
        prompt = prompt + "Source : " + source + "\n"
    prompt = prompt + "Titre : " + titre + "\n"
    prompt = prompt + "Texte : " + texte_court + "\n\n"

    if avec_rag:
        prompt = prompt + "Preuves trouvees sur le web concernant cette affirmation :\n"
        prompt = prompt + preuves + "\n\n"
        prompt = prompt + "En te basant sur ces preuves, verifie si l'affirmation de cet article est vraie ou fausse.\n"
    else:
        prompt = prompt + "En te basant sur ce que tu connais déja, verifie si l'affirmation de cet article est vraie ou fausse.\n"
    
    prompt = prompt + "Reponds uniquement par un seul mot : \"fake\" ou \"real\"."

    return prompt


def construire_prompt(titre, texte, source, preuves, avec_rag):
    texte_court = texte[:800]

    prompt = "Here is an article to verify.\n\n"
    if source != "":
        prompt = prompt + "Source: " + source + "\n"
    prompt = prompt + "Title: " + titre + "\n"
    prompt = prompt + "Text: " + texte_court + "\n\n"

    if avec_rag:
        prompt = prompt + "Evidence found on the web about this claim:\n"
        prompt = prompt + preuves + "\n\n"
        prompt = prompt + "Based on this evidence, verify whether the claim in this article is true or false.\n"
    else:
        prompt = prompt + "Based on what you already know, verify whether the claim in this article is true or false.\n"

    prompt = prompt + "Reply with one word only: \"fake\" or \"real\"."

    return prompt


def extraire_verdict(reponse_llm):
    texte = reponse_llm.lower().strip()

    contient_fake = bool(re.search(r"\bfake\b", texte))
    contient_real = bool(re.search(r"\breal\b", texte))

    # uniquemnt fake
    if contient_fake and not contient_real:
        return "fake"

    # uniquemnt real
    if contient_real and not contient_fake:
        return "real"

    # sinon
    return "indetermine"

In [ ]:
def demander_verdict(titre, texte, source, preuves, avec_rag):

    prompt = construire_prompt(titre, texte, source, preuves, avec_rag)

    reponse = ollama.chat(
        model=m,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0}
    )

    contenu = reponse["message"]["content"]

    return extraire_verdict(contenu), contenu


In [ ]:

FICHIER_PREUVES = chemin_out + "preuves_150.csv"

def sauver_preuve(id_article, requete, preuves, domaines):
    nouveau = not os.path.exists(FICHIER_PREUVES)
    with open(FICHIER_PREUVES, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nouveau:
            w.writerow(["id", "requete", "Preuves", "domaine", "date_recherche"])
        w.writerow([id_article, requete, preuves, " | ".join(domaines), datetime.date.today().isoformat()])

In [ ]:
def tester_contrefactuel(article, source, avec_rag, creer_preuves):

    titre = article.get("title", "")
    texte = article.get("text", "")
    vraie_source = article.get("domaine", "source inconnue")
    vrai_label = article.get("label", "")

    # RAG or no-RAG
    if avec_rag:
        if creer_preuves:
            requete = titre if titre else texte[:100]
            preuves, domaines_rag = chercher_preuves_2(requete)

            # Save in memory
            article["preuves"] = preuves

            # Save to the CSV file
            sauver_preuve(article["id"], requete, preuves, domaines_rag)
            print(f"RAG CREATED for : {article.get('id', '')}")

        else:
            # reuse the same evidence
            preuves = article["preuves"]
            print(f"RAG REUSED for : {article.get('id', '')}")

    else:
        preuves = None

    # A single verdict, with the source passed as a parameter
    verdict, reponse_brute = demander_verdict(titre, texte, source, preuves, avec_rag)

    return {
        "id": article.get("id", ""),
        "vraie_source": vraie_source,
        "vrai_label": vrai_label,
        "source_testee": source,
        "verdict": verdict,
        "reponse_brute": reponse_brute
    }

In [ ]:
tirage = [{k: r[k] for k in "ABCD"} for r in lire_csv(chemin_tirage)]

for i, t in enumerate(tirage, 1):
    print(i, t)

print(len(tirage), "draws loaded")

In [ ]:
def lancer_test_tirage(echantillon, chemin_sortie, sources, avec_rag, creer_preuves):
    colonnes = ["id", "vraie_source", "vrai_label",
                "source_A", "verdict_A", "reponse_A",
                "source_B", "verdict_B", "reponse_B",
                "source_C", "verdict_C", "reponse_C",
                "source_D", "verdict_D", "reponse_D",
                "verdict_S", "reponse_S"]

    lignes = []
    for i, article in enumerate(echantillon, 1):
        ligne = {
            "id": article.get("id"),
            "vraie_source": article.get("domaine", "source inconnue"),
            "vrai_label": article.get("label"),
        }

        # Evidence handling (only used when avec_rag is True):
        # S (no source): creates new evidence if creer_preuves is True,
        # otherwise reuses article["preuves"] (loaded from preuves_150-2.csv).
        # A, B, C, D: creer_preuves is always False, so they use article["preuves"],
        
        res = tester_contrefactuel(article, "", avec_rag, creer_preuves)
        ligne["verdict_S"] = res["verdict"]
        ligne["reponse_S"] = res["reponse_brute"]

        for k in ["A", "B", "C", "D"]:
            res = tester_contrefactuel(article, sources[k], avec_rag, False)
            ligne[f"source_{k}"] = sources[k]
            ligne[f"verdict_{k}"] = res["verdict"]
            ligne[f"reponse_{k}"] = res["reponse_brute"]

        lignes.append(ligne)
        print(f"[{i}/{len(echantillon)}] OK")

    with open(chemin_sortie, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=colonnes)
        w.writeheader()
        w.writerows(lignes)

In [ ]:
# run with the English prompt
PREFIXE = "en_brute_800_carac"
MODS = ["llama3.2:1b", "llama3.2:3b", "gemma3:4b", "qwen2.5:7b","phi4-mini"]

# preuve = False because we want to load the RAG evidence from the preuves_150-2.csv file retrieved earlier
preuve = False

for m in MODS:
    nom = m.replace(":", "_")

    for avec_rag in (True, False):
        mode = "RAG" if avec_rag else "WRAG"
        print("====================", mode, ":", m, "====================")

        for num, sources in enumerate(tirage, 1):
            fichier = chemin_out + f"{PREFIXE}_tirage{num}_{nom}_{mode}.csv"

            if os.path.exists(fichier):
                print(f"Draw {num} already done, skipping")
                continue

            print(f"Tirage {num} -> {sources}")
            lancer_test_tirage(echantillon=articles,
                               chemin_sortie=fichier,
                               sources=sources,
                               avec_rag=avec_rag,
                               creer_preuves=preuve)

In [ ]:
# run with the French prompt
MODS = ["llama3.2:1b", "llama3.2:3b", "gemma3:4b", "qwen2.5:7b","phi4-mini"]

# preuve = False because we want to load the RAG evidence from the preuves_150-2.csv file retrieved earlier
preuve = False

for m in MODS:
    nom = m.replace(":", "_")

    for avec_rag in (True, False):
        mode = "RAG" if avec_rag else "WRAG"
        print("====================", mode, ":", m, "====================")

        for num, sources in enumerate(tirage, 1):
            fichier = chemin_out + f"tirage{num}_{nom}_{mode}.csv"

            if os.path.exists(fichier):
                print(f"Tirage {num} déjà fait, on saute")
                continue

            print(f"Tirage {num} -> {sources}")
            lancer_test_tirage(echantillon=articles,
                               chemin_sortie=fichier,
                               sources=sources,
                               avec_rag=avec_rag,
                               creer_preuves=preuve)

# **Other experiments**

We run the next cell to repeat the whole experiment with new evidence searched on the web, instead of reusing the evidence from the preuves_150-2.csv file.
The evidence is searched only once, during the first draw with RAG. It is then reused for all the other draws and all the models, so every model sees exactly the same evidence and the comparison stays fair.
The new evidence is saved in the preuves_150.csv file (FICHIER_PREUVES = chemin_out + "preuves_150.csv"), in the out/ folder.

In [ ]:
MODS = ["llama3.2:1b", "llama3.2:3b", "gemma3:4b", "qwen2.5:7b","phi4-mini"]
PREFIXE_NEW = "en_new_evidence"

preuve = True # True only for the very first run with RAG

for m in MODS:
    nom = m.replace(":", "_")

    for avec_rag in (True, False):
        mode = "RAG" if avec_rag else "WRAG"
        print("====================", mode, ":", m, "====================")

        for num, sources in enumerate(tirage, 1):
            fichier = chemin_out + f"{PREFIXE_NEW}_tirage{num}_{nom}_{mode}.csv"

            print(f"Tirage {num} -> {sources}")
            lancer_test_tirage(echantillon=articles,
                               chemin_sortie=fichier,
                               sources=sources,
                               avec_rag=avec_rag,
                               creer_preuves=preuve)
            
            preuve = False


###   Cell that generates the tirages_7.csv file:
This cell randomly picks 7 groups of four news sources, one from each category, and saves them to a file, making sure that no source is used twice.

In [ ]:
# This cell generates the CSV file containing the 7 draws for each category
random.seed(42)

N = 7  # number of draws = number of domains in B

domB = [d["domaine_principal"] for d in catB]
domA = [d["domaine_principal"] for d in catA]
domC = [d["domaine_principal"] for d in catC]
domD = [d["domaine_principal"] for d in catD]

assert len(domB) == N, f"B contains {len(domB)} domains, not {N}"

random.shuffle(domB)
tirA = random.sample(domA, N)
tirC = random.sample(domC, N)
tirD = random.sample(domD, N)

TIRAGES = [{"A": tirA[i], "B": domB[i], "C": tirC[i], "D": tirD[i]} for i in range(N)]

with open(chemin_out + "tirages_7.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["tirage", "A", "B", "C", "D"])
    w.writeheader()
    for i, t in enumerate(TIRAGES, 1):
        w.writerow({"tirage": i, **t})

for i, t in enumerate(TIRAGES, 1):
    print(i, t)

tous = [d for t in TIRAGES for d in t.values()]
assert len(tous) == 28, f"{len(tous)} domaines au lieu de 28"
assert len(set(tous)) == 28, "there are repeated domains"
print("OK : 7 draws x 4 domains = 28 distinct domains")

##  Experiment showing what the LLMs know about the domains in the tirages_7.csv file:
This experiment asks each language model whether it knows each news source used in our draws, and records how it describes the source and how reliable it thinks it is.

In [ ]:
# asks the models whether they know the sources selected in our draws
# version that also saves the raw LLM responses

tirage_chemin = "data/tirages_7.csv"


liste_domaines = ["fool.com", "buzzfeednews.com", "latimes.com", "statista.com",
                  "blogspot.com", "bgr.com", "cbs8.com", "theweek.co.uk",
                  "ibtimes.co.in", "metro.news", "dailybuzzlive.com"]


with open(tirage_chemin, newline="", encoding="utf-8") as f:
    for ligne in csv.DictReader(f):
        for cat in "ABCD":
            if ligne[cat] not in liste_domaines:
                liste_domaines.append(ligne[cat])

print(len(liste_domaines), "domaines")


MODS = ["llama3.2:1b", "llama3.2:3b", "gemma3:4b", "qwen2.5:7b", "phi4-mini"]

# French version
def question_domaine_fr(domaine):
    return (
        f"Connais-tu le média {domaine} ?\n\n"
        "Réponds exactement dans ce format :\n"
        "Connaissance : oui ou non\n"
        f"Description : une phrase qui décrit le média {domaine}, ou « aucune » si tu ne le connais pas\n"
        "Fiabilité : fiable, peu fiable, incertain ou inconnu"
    )

# English version
def question_domaine(domaine):
    return (
        f"Do you know the media {domaine}?\n\n"
        "Answer exactly in this format:\n"
        "Knowledge: yes or no\n"
        f"Description: one sentence describing the media {domaine}, or \"none\" if you don't know it\n"
        "Reliability: reliable, unreliable, uncertain or unknown"
    )

for modele in MODS:
    nom = modele.replace(":", "_")
    chemin = chemin_out + f"connaissance_{nom}.csv"

    with open(chemin, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["modele", "domaine", "reponse", "reponse_brute"])

        for domaine in liste_domaines:
            rep = ollama.chat(
                model=modele,
                messages=[{"role": "user", "content": question_domaine(domaine)}],
                options={"temperature": 0}
            )
            brut = rep["message"]["content"]
            w.writerow([modele, domaine, brut.strip(), brut])
            f.flush()

    print("OK :", chemin)